# ARC-AGI-2 | CPU Transforms and Honest Failures

**What can a tiny, transparent CPU solver actually solve?** This notebook tries eight rotations/reflections,
then adds a demonstration-consistent color lookup. No model downloads, GPU, internet or private dependencies.

The reference is intentionally narrow. Verified Kaggle and local runs solved **7 of 1,076 public training queries** with geometry
and **11 of 1,076** with color mapping. This is a useful floor and a tested scoring scaffold, not a competitive solver.
The live tables below are the authoritative result of this run.

## Fixed experiment

Rules are inferred independently for each task from its demonstration input/output pairs. Every query prediction
is frozen **before opening the training-query solution file**. Geometry candidates come first in a fixed order;
recoloring cannot displace an existing geometry guess. Duplicate guesses are removed; unseen color mappings abstain.

We compare query exact match with at most two guesses, the mean query score per task, and whole-task success.
These are explicitly public-training reference metrics, **not official competition evaluation or leaderboard scores**.
The public training collection was used previously for an atlas, so this is not a pristine held-out benchmark.
No evaluation/test files or submission are used. No raw task grids, predictions or task-level results are exported.


In [ ]:
import hashlib
import json
import os
import time
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT = Path(os.environ.get('NB_OUTPUT', '/kaggle/working'))
OUTPUT.mkdir(parents=True, exist_ok=True)
roots = [Path(os.environ['ARC_DATA_ROOT'])] if os.environ.get('ARC_DATA_ROOT') else [
    Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2'),
    Path('/kaggle/input/arc-prize-2026-arc-agi-2')]
DATA = next(p for p in roots if (p / 'arc-agi_training_challenges.json').is_file())
challenge_path = DATA / 'arc-agi_training_challenges.json'
solution_path = DATA / 'arc-agi_training_solutions.json'
tasks = json.loads(challenge_path.read_text())

## A demonstration-only reference solver

In [ ]:
def transform(grid, index):
    array = np.asarray(grid, dtype=np.int8)
    return np.rot90(np.fliplr(array) if index >= 4 else array, index % 4)


def infer_rules(demonstrations, recolor=False):
    """Learn only from demonstrations. Ordering is fixed before reading query answers."""
    candidates = []
    for index in range(8):
        mapping = {}
        valid = True
        for pair in demonstrations:
            x = transform(pair['input'], index)
            y = np.asarray(pair['output'], dtype=np.int8)
            if x.shape != y.shape:
                valid = False
                break
            if not recolor:
                if not np.array_equal(x, y):
                    valid = False
                    break
            else:
                for a, b in zip(x.flat, y.flat):
                    if int(a) in mapping and mapping[int(a)] != int(b):
                        valid = False
                        break
                    mapping[int(a)] = int(b)
                if not valid:
                    break
        if valid:
            candidates.append((index, mapping if recolor else None))
    return candidates


def predict(demonstrations, query, recolor=False, limit=2):
    # Keep geometry-only rules first, so recoloring cannot displace an existing guess.
    candidates = infer_rules(demonstrations)
    if recolor:
        candidates += infer_rules(demonstrations, recolor=True)
    predictions = []
    for index, mapping in candidates:
        result = transform(query, index)
        if mapping is not None:
            if not set(np.unique(result)).issubset(mapping):
                continue  # Never infer an unseen color mapping from a query answer.
            lut = np.arange(10, dtype=np.int8)
            for a, b in mapping.items():
                lut[a] = b
            result = lut[result]
        if not any(np.array_equal(result, p) for p in predictions):
            predictions.append(result)
        if len(predictions) == limit:
            break
    return predictions

## Synthetic controls

In [ ]:
# Synthetic known-answer and failure controls.
asymmetric = [[1, 2, 3], [4, 5, 6]]
demo = [{'input': asymmetric, 'output': np.rot90(asymmetric).tolist()}]
assert np.array_equal(predict(demo, asymmetric)[0], np.rot90(asymmetric))
recolor_demo = [{'input': [[1, 1], [1, 1]], 'output': [[2, 2], [2, 2]]}]
assert not predict(recolor_demo, [[1, 1], [1, 1]])
assert np.array_equal(predict(recolor_demo, [[1, 1], [1, 1]], True)[0], np.full((2, 2), 2))
assert not predict(recolor_demo, [[3, 3], [3, 3]], True)
assert not infer_rules([{'input': [[1, 1]], 'output': [[2, 3]]}], True)
assert not predict([{'input': [[1]], 'output': [[1, 1]]}], [[1]])
assert len(predict([{'input': [[1]], 'output': [[1]]}], [[1]], True)) == 1
print('Known-answer controls: PASS')

## Freeze predictions, then score

In [ ]:
# Prediction phase has no access to solutions: only tasks and their demonstration outputs.
started = time.perf_counter()
guesses = {}
for task_id, task in tasks.items():
    for variant, recolor in [('geometry', False), ('geometry_plus_color', True)]:
        guesses[(task_id, variant)] = [predict(task['train'], q['input'], recolor) for q in task['test']]
prediction_seconds = time.perf_counter() - started

# Only after every prediction is frozen do we open public training-query answers.
solutions = json.loads(solution_path.read_text())
rows = []
for task_id, task in tasks.items():
    assert len(solutions[task_id]) == len(task['test'])
    shape_changed = any(np.shape(p['input']) != np.shape(p['output']) for p in task['train'])
    for variant in ['geometry', 'geometry_plus_color']:
        first, top2, coverage, multiple = [], [], [], []
        for predictions, answer in zip(guesses[(task_id, variant)], solutions[task_id]):
            first.append(bool(predictions) and np.array_equal(predictions[0], answer))
            top2.append(any(np.array_equal(p, answer) for p in predictions))
            coverage.append(bool(predictions))
            multiple.append(len(predictions) > 1)
        rows.append({'variant': variant, 'shape_group': 'shape-changing' if shape_changed else 'same-shape',
                     'queries': len(first), 'top1_hits': sum(first), 'top2_hits': sum(top2),
                     'covered_queries': sum(coverage), 'multiple_queries': sum(multiple),
                     'task_mean_top2': float(np.mean(top2)), 'all_queries_correct': all(top2)})
results = pd.DataFrame(rows)
summary_rows = []
for variant, group in results.groupby('variant', sort=False):
    queries = int(group.queries.sum())
    summary_rows.append({'variant': variant, 'tasks': len(group), 'queries': queries,
        'top1_hits': int(group.top1_hits.sum()), 'top2_hits': int(group.top2_hits.sum()),
        'top1_query_rate': float(group.top1_hits.sum() / queries),
        'top2_query_rate': float(group.top2_hits.sum() / queries),
        'task_mean_top2': float(group.task_mean_top2.mean()),
        'all_queries_correct_tasks': int(group.all_queries_correct.sum()),
        'coverage_rate': float(group.covered_queries.sum() / queries),
        'multiple_guess_rate': float(group.multiple_queries.sum() / queries)})
scores = pd.DataFrame(summary_rows)
cohorts = results.groupby(['variant', 'shape_group']).agg(tasks=('queries', 'size'),
    queries=('queries', 'sum'), top2_hits=('top2_hits', 'sum'), covered_queries=('covered_queries', 'sum')).reset_index()
cohorts['top2_query_rate'] = cohorts.top2_hits / cohorts.queries
assert len(tasks) == 1000 and scores.queries.eq(1076).all()
assert scores.iloc[1].top2_hits >= scores.iloc[0].top2_hits
print(scores.to_string(index=False))
print(cohorts.to_string(index=False))

## Coverage and failure cohorts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
scores.set_index('variant')[['top1_query_rate', 'top2_query_rate', 'coverage_rate']].plot.bar(
    ax=axes[0], color=['#34699a', '#3a8464', '#cb8838'], rot=0)
axes[0].set(ylabel='Fraction of public training queries', xlabel='', title='Exact match versus coverage')
axes[0].tick_params(axis='x', labelsize=8)
cohorts.pivot(index='shape_group', columns='variant', values='top2_query_rate').plot.bar(
    ax=axes[1], color=['#34699a', '#3a8464'], rot=0)
axes[1].set(ylabel='Top-2 query exact-match fraction', xlabel='', title='Where simple rules fail')
fig.savefig(OUTPUT / 'solver_results.png', dpi=150)
plt.show()
scores.to_csv(OUTPUT / 'scores.csv', index=False)
cohorts.to_csv(OUTPUT / 'cohorts.csv', index=False)
summary = {'observed_utc': datetime.now(timezone.utc).isoformat(), 'competition': 'arc-prize-2026-arc-agi-2',
    'score_type': 'public_training_query_reference_not_leaderboard',
    'challenge_sha256': hashlib.sha256(challenge_path.read_bytes()).hexdigest(),
    'solutions_sha256': hashlib.sha256(solution_path.read_bytes()).hexdigest(),
    'prediction_seconds': prediction_seconds, 'known_answer_controls': 'PASS', 'scores': summary_rows,
    'submission_created': False, 'test_or_evaluation_read': False, 'leaderboard_score': None}
(OUTPUT / 'summary.json').write_text(json.dumps(summary, indent=2, allow_nan=False))
print('Run checks: PASS')

## What to reuse, and where it fails

Reuse `infer_rules()` and `predict()` as a compact reference interface and the separation between prediction and
scoring. `limit=2` is a guess budget, not a claim of two independent models. Zero candidates produce an abstention.

This family cannot resize by scaling/cropping, segment objects, count components or execute relational rules.
Rotations may swap height and width, but most shape-changing tasks require much more: the measured shape-changing
cohort has zero hits. Color mappings are not required to be bijections, and unknown query colors are never guessed.
Near-perfect precision among the very few covered queries does not imply useful overall coverage.

Next controlled extension: add **one** object-based rule family, preserve prediction-before-scoring, and report
coverage as well as accuracy. Any further tuning on this collection requires a genuinely separate evaluation set.

## Sources

- [Official competition and input](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-2/data),
  [rules](https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-2/rules), observed 2026-09-08.
- [Previous task atlas](https://www.kaggle.com/code/muelsyse111/arc-agi-2-task-atlas-and-grid-viewer): descriptive
  structure analysis, not solver evidence. This notebook adds executable inference and query-level evaluation.
- NumPy `rot90`, `fliplr`, array comparison and indexing implement the primitive operations.

Original reference code prepared with AI assistance. Official inputs remain on Kaggle; no outside agent was copied.
Data files were listed as created 2026-03-10, not newly released today. Consult competition rules before reuse.
